# Fig. S1 — Supplementary panels (S1A, S1c)

Companion to `20260417_Figure1_Nature.ipynb`. Uses the same `adata_RNA_CITE.h5ad` and color scheme.

- **S1A** — Experimental QC: cells per sample, per-sample HLA-I distribution (sort reproducibility), IFNγ-response signature per condition.
- **S1c** — CITE-seq markers higher in HLA-high vs HLA-low (HLA-DR, CD274, CD47, CD49f, CD58), 5-panel violin row.

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from matplotlib import rcParams

rcParams['pdf.fonttype'] = 42
rcParams['ps.fonttype'] = 42
rcParams['pdf.use14corefonts'] = True
warnings.filterwarnings(action='ignore')
sc.set_figure_params(figsize=[4, 4], fontsize=12, dpi=100, frameon=False)

In [ ]:
COND_COLORS = {'Low': '#e1812c', 'Control': '#3274a1', 'High': '#3a923a'}
COND_ORDER = ['Low', 'Control', 'High']

OUT_DIR = '../figures/nature_figures/FigS1'
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(name, formats=('pdf', 'png')):
    for fmt in formats:
        plt.savefig(os.path.join(OUT_DIR, f'{name}.{fmt}'), bbox_inches='tight', dpi=300)
    print(f'Saved: {name}')

In [ ]:
BASE = '/home/wangh256/hanchen/Pert_PG/perturb-me/PerturbME_transfer/PerturbCITE_ICR/202008_full_exp'
adata = sc.read_h5ad(os.path.join(BASE, 'adata_RNA_CITE.h5ad'))
adata.obs['Condition'] = pd.Categorical(adata.obs['Condition'].astype(str), categories=COND_ORDER)
condition_arr = adata.obs['Condition'].to_numpy().astype(str)
print(f'Loaded: {adata.shape[0]:,} cells × {adata.shape[1]:,} features')
print({c: int((condition_arr == c).sum()) for c in COND_ORDER})

def extract_expr(feature):
    x = adata[:, feature].X
    return x.toarray().flatten() if hasattr(x, 'toarray') else x.flatten()

## Fig S1A — Experimental QC

### S1A-i — Cells per sample, split by condition

In [ ]:
samp_cond = adata.obs.groupby(['Sample', 'Condition'], observed=True).size().reset_index(name='n_cells')
samp_cond['Sample'] = samp_cond['Sample'].astype(str)
# Sort: Low samples, then Control, then High; within each condition by descending count
samp_cond['cond_order'] = samp_cond['Condition'].map({'Low': 0, 'Control': 1, 'High': 2})
samp_cond = samp_cond.sort_values(['cond_order', 'n_cells'], ascending=[True, False])

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(samp_cond['Sample'], samp_cond['n_cells'],
       color=[COND_COLORS[c] for c in samp_cond['Condition']],
       edgecolor='white', linewidth=0.5)
ax.set_ylabel('Cells (n)')
ax.set_xlabel('Sample (10X channel)')
plt.xticks(rotation=60, ha='right', fontsize=9)
ax.set_ylim(0, samp_cond['n_cells'].max() * 1.1)
# Legend
handles = [plt.Rectangle((0, 0), 1, 1, color=COND_COLORS[c]) for c in COND_ORDER]
ax.legend(handles, COND_ORDER, title='Condition', frameon=False, loc='upper right')
sns.despine(); ax.grid(False)
plt.tight_layout()
save_fig('S1A_i_cells_per_sample')
plt.show()

### S1A-ii — Per-sample CITE-HLA_A distribution (sort gate reproducibility)

In [ ]:
hla_protein = extract_expr('CITE-HLA_A')
df_ridge = pd.DataFrame({
    'HLA-A,B,C protein': hla_protein,
    'Sample': adata.obs['Sample'].astype(str).values,
    'Condition': condition_arr,
})
# Order samples by condition then mean expression
samp_means = df_ridge.groupby(['Sample', 'Condition'], observed=True)['HLA-A,B,C protein'].mean().reset_index()
samp_means['cond_order'] = samp_means['Condition'].map({'Low': 0, 'Control': 1, 'High': 2})
samp_order = samp_means.sort_values(['cond_order', 'HLA-A,B,C protein'], ascending=[True, True])['Sample'].tolist()
samp_to_cond = dict(zip(samp_means['Sample'], samp_means['Condition']))

fig, ax = plt.subplots(figsize=(6, 5))
x_grid = np.linspace(df_ridge['HLA-A,B,C protein'].min(), df_ridge['HLA-A,B,C protein'].max(), 200)
row_height = 1.0
for i, samp in enumerate(samp_order):
    vals = df_ridge.loc[df_ridge['Sample'] == samp, 'HLA-A,B,C protein'].values
    if len(vals) < 10:
        continue
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(vals, bw_method=0.2)
    dens = kde(x_grid)
    dens = dens / dens.max() * 0.95
    color = COND_COLORS[samp_to_cond[samp]]
    ax.fill_between(x_grid, i, i + dens, color=color, alpha=0.75, linewidth=0.5, edgecolor='white')
    ax.plot(x_grid, i + dens, color='black', linewidth=0.3)

ax.set_yticks([i + 0.4 for i in range(len(samp_order))])
ax.set_yticklabels(samp_order, fontsize=8)
ax.set_xlabel('HLA-A,B,C protein (CITE-seq)')
ax.set_ylabel('Sample')
ax.set_ylim(-0.2, len(samp_order))
handles = [plt.Rectangle((0, 0), 1, 1, color=COND_COLORS[c]) for c in COND_ORDER]
ax.legend(handles, COND_ORDER, title='Condition', frameon=False, loc='lower right')
sns.despine(); ax.grid(False)
plt.tight_layout()
save_fig('S1A_ii_per_sample_HLA_ridgeline')
plt.show()

### S1A-iii — IFNγ-response signature per condition

Mean log-normalized expression of canonical IFNγ-induced genes (Hallmark INTERFERON_GAMMA_RESPONSE core members) per cell, summarized by condition.

In [ ]:
IFNG_GENES = ['STAT1', 'IRF1', 'B2M', 'TAP1', 'TAP2', 'TAPBP',
              'HLA-A', 'HLA-B', 'HLA-C', 'HLA-E',
              'GBP1', 'IFITM1', 'ISG15', 'IFIT1', 'IFIT3', 'MX1', 'OAS1', 'WARS']
present = [g for g in IFNG_GENES if g in adata.var_names]
print(f'IFNγ signature genes used ({len(present)}/{len(IFNG_GENES)}): {present}')

sc.tl.score_genes(adata, gene_list=present, score_name='IFNG_score', use_raw=False)

df_score = pd.DataFrame({
    'IFNγ score': adata.obs['IFNG_score'].values,
    'Condition': condition_arr,
})

fig, ax = plt.subplots(figsize=(3.5, 3.5))
sns.violinplot(data=df_score, x='Condition', y='IFNγ score',
               order=COND_ORDER, palette=COND_COLORS, cut=0, inner='box',
               saturation=0.85, ax=ax)
for coll in ax.collections:
    coll.set_alpha(0.75)
ax.set_xlabel('')
ax.set_ylabel('IFNγ-response score')
sns.despine(); ax.grid(False)
plt.tight_layout()
save_fig('S1A_iii_IFNG_signature_by_condition')
plt.show()

print('\nMean IFNγ score by condition:')
print(df_score.groupby('Condition', observed=True)['IFNγ score'].agg(['mean', 'median', 'std']).round(3))

## Fig S1c — CITE-seq markers across conditions

HLA-DR (CITE-HLA_D), CD274 (PD-L1), CD47, CD49f, CD58 — reported higher in HLA-high vs HLA-low.

In [ ]:
S1C_MARKERS = [
    ('CITE-HLA_D', 'HLA-DR'),
    ('CITE-CD274', 'CD274 (PD-L1)'),
    ('CITE-CD47',  'CD47'),
    ('CITE-CD49f', 'CD49f'),
    ('CITE-CD58',  'CD58'),
]

long_rows = []
for feat, label in S1C_MARKERS:
    vals = extract_expr(feat)
    long_rows.append(pd.DataFrame({
        'Expression': vals,
        'Condition': condition_arr,
        'Marker': label,
    }))
df_s1c = pd.concat(long_rows, ignore_index=True)
df_s1c['Marker'] = pd.Categorical(df_s1c['Marker'], categories=[lbl for _, lbl in S1C_MARKERS])
print(df_s1c.groupby(['Marker', 'Condition'], observed=True)['Expression'].agg(['mean', 'median']).round(3))

### S1c — Violin row of CITE markers across conditions

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3.5), sharey=False)
for ax, (feat, label) in zip(axes, S1C_MARKERS):
    sub = df_s1c[df_s1c['Marker'] == label]
    sns.violinplot(data=sub, x='Condition', y='Expression', ax=ax,
                   order=COND_ORDER, palette=COND_COLORS, cut=0, inner='box',
                   saturation=0.85)
    for coll in ax.collections:
        coll.set_alpha(0.75)
    ax.set_title(label)
    ax.set_xlabel('')
    ax.set_ylabel('CITE expression')
sns.despine()
for ax in axes: ax.grid(False)
plt.tight_layout()
save_fig('S1c_violin_row')
plt.show()